In [ ]:
import torch
import torch.nn as nn
import gymnasium as gym
import numpy as np
import mujoco

from torch.utils.tensorboard import SummaryWriter
import time
from lerobot.datasets import LeRobotDataset
from lerobot.processor.normalize_processor import NormalizerProcessorStep
from lerobot.processor.normalize_processor import UnnormalizerProcessorStep
from torch.utils.data import DataLoader, Subset

from cobot_BC_env import CobotBcEnv
from Behavior_Cloning import BehaviorCloning, BcDataset
from record_expert import RecordExpert
from pick_place_expert import PickPlaceExpert

In [ ]:
class DAgger:
    def __init__(self, dataset, model_checkpoint = None):
        import shutil
        self.dataset_name =dataset + "_dag"
        self.dataset_writer = LeRobotDataset.resume(
            root="./datasets/" + dataset,
            repo_id="aaron/" + dataset,
        )
        
        src_root = f"./datasets/{dataset}"
        dst_root = f"./datasets/{self.dataset_name}"
        shutil.copytree(src_root, dst_root)

        self.env = CobotBcEnv(render_mode= None)
        self.model = BehaviorCloning(self.env, BcDataset(name = dataset), hidden_dims=(256,256,256))
        self.expert = PickPlaceExpert(self.env)
        if model_checkpoint:
            model_checkpoint = torch.load("checkpoints/" + model_checkpoint, map_location=torch.device('cpu')) 
            self.model.policy.load_state_dict(model_checkpoint["policy"])
            self.model.policy_optimizer.load_state_dict(model_checkpoint["policy_optimizer"])


    @torch.no_grad()
    def add_episode(self, n_steps=300, beta = 0.0):
        self.env.reset()
        self.model.policy.eval()
        rng = np.random.default_rng()
        for i in range(n_steps):
            raw_state = self.env.get_observation()
            state_dict = {"observation.state": torch.from_numpy(raw_state).float().to('cpu')}
            state = self.model.dataset.normalizer(state_dict)["observation.state"]

            expert_action = np.asarray(
                self.expert(raw_state), dtype=np.float32
            )
            
            self.dataset_writer.add_frame({
                        "observation.state": raw_state,
                        "action": expert_action,
                        "task": "pick up the cube and place it on the disk",
                    })
            
            if rng.random() > beta : 

                policy_normalized_action = self.model.policy(state)
                action_dict = {"action": policy_normalized_action}
                action = self.model.dataset.unnormalizer(action_dict)["action"].cpu().numpy()
            else:
                action = expert_action

            self.env.apply_action(action)

            if self.env.render_mode == "human":
                self.env.render()
        self.dataset_writer.save_episode()


    def create_dataset(self, n_episodes, n_steps=300, beta=0.0):
        while self.dataset_writer.num_episodes < n_episodes:
            self.add_episode(n_steps, beta)
            print(str(self.dataset_writer.num_episodes))
        self.dataset_writer.finalize()
        print('dataset created')

    def refresh_dataset(self):
        old_normalizer = self.model.dataset.normalizer
        old_unnormalizer = self.model.dataset.unnormalizer

        self.model.dataset = BcDataset(self.dataset_name)

        self.model.dataset.normalizer = old_normalizer
        self.model.dataset.unnormalizer = old_unnormalizer

    def iterative_training(self, 
                           n_itr = 5,
                           n_eps_add = 10,
                           n_epochs = 3,
                           n_steps = 300,
                           beta = 0.0,
                           rollout = True,
                           ):
        
        n_eps = self.dataset_writer.num_episodes 

        for _ in range(n_itr):
            n_eps += n_eps_add
            self.dataset_writer = LeRobotDataset.resume(
                repo_id=f"aaron/{self.dataset_name}",
                root=f"./datasets/{self.dataset_name}",
            )
            self.create_dataset(n_eps, n_steps, beta)
            self.refresh_dataset()

            self.model.train(n_epochs)

            if rollout:
                self.model.rollout

In [ ]:
dagger = DAgger(
        'dagger_512_3_500',
        model_checkpoint = "dagger_256_3_v1.pt"    
    )

In [ ]:
dagger.iterative_training(n_itr=10, n_eps_add=50, n_epochs=2, n_steps=300, beta=0.0)

In [ ]:
for i in range(3):
    dagger.model.rollout()

In [ ]:
torch.save({
    "policy": dagger.model.policy.state_dict(),
    "policy_optimizer": dagger.model.policy_optimizer.state_dict(),
}, "checkpoints/dagger_256_3_v2.pt")